# 面试题：Agent 怎样判断 need-tool？

可复述的答案：先判断回答是否依赖实时、私有、可验证或有副作用的状态，再检查工具授权和参数完整性。知识解释可直接回答；查询与写入通常要工具；工具不可用或参数缺失时必须返回不确定性或追问，不能编造。模型分类只产生候选，服务端授权和参数校验才是最终门禁。

## 真实案例

电商客服处理规则解释、物流查询、修改地址、库存查询、文案生成和取消订单共 6 条请求。

## 基线

基线按“是否出现查/取消”关键词决定调用。

## 结果解读

手写决策显式输出时效、私有、写入、授权和参数条件。

## 失败案例

“如何取消订单”包含取消却是知识问题，关键词策略会误调用。

In [1]:
requests = [{'id':'C01','text':'退货规则是什么','fresh':False,'private':False,'write':False,'authorized':True,'slots':True}, {'id':'C02','text':'查订单 A12 的物流','fresh':True,'private':True,'write':False,'authorized':True,'slots':True}, {'id':'C03','text':'把订单 A13 地址改到北京','fresh':False,'private':True,'write':True,'authorized':True,'slots':True}, {'id':'C04','text':'这款耳机现在有库存吗','fresh':True,'private':False,'write':False,'authorized':True,'slots':True}, {'id':'C05','text':'写一段耳机文案','fresh':False,'private':False,'write':False,'authorized':True,'slots':True}, {'id':'C06','text':'取消订单 A14','fresh':False,'private':True,'write':True,'authorized':False,'slots':True}]  # 构造六条包含时效、私有与副作用字段的客服请求。
print('教学实验输入：id | 请求 | 实时 | 私有 | 写入 | 授权')  # 输出原始字段以便检查决策来源。
for row in requests:  # 逐条展示客服事件。
    print(row['id'], row['text'], row['fresh'], row['private'], row['write'], row['authorized'])  # 输出当前请求的完整判断特征。

教学实验输入：id | 请求 | 实时 | 私有 | 写入 | 授权
C01 退货规则是什么 False False False True
C02 查订单 A12 的物流 True True False True
C03 把订单 A13 地址改到北京 False True True True
C04 这款耳机现在有库存吗 True False False True
C05 写一段耳机文案 False False False True
C06 取消订单 A14 False True True False


In [2]:
baseline = {row['id']:('调用工具' if ('查' in row['text'] or '取消' in row['text']) else '直接回答') for row in requests}  # 用脆弱关键词建立是否调用工具的基线。
print('关键词基线:', baseline)  # 输出 baseline 的逐请求动作。
print('基线问题：它无法区分“如何取消”的说明问题和真正取消，也不知道权限。')  # 明确基线漏掉的安全条件。

关键词基线: {'C01': '直接回答', 'C02': '调用工具', 'C03': '直接回答', 'C04': '直接回答', 'C05': '直接回答', 'C06': '调用工具'}
基线问题：它无法区分“如何取消”的说明问题和真正取消，也不知道权限。


In [3]:
def decide_need_tool(row):  # 定义独立于措辞的 need-tool 控制逻辑。
    needs_state = row['fresh'] or row['private'] or row['write']  # 判断回答是否依赖外部权威状态或副作用。
    if not needs_state:  # 处理不依赖工具的知识与创作请求。
        return '直接回答', '无实时或私有状态'  # 返回 no-tool 决策及可解释原因。
    if not row['authorized']:  # 拒绝未经授权的私有或写操作。
        return '拒绝并说明授权', '缺少授权'  # 防止 Router 以调用工具绕过权限。
    if not row['slots']:  # 缺少关键参数时先获取信息。
        return '追问参数', '缺少必填字段'  # 防止模型猜测订单等标识。
    return '调用工具', '需要权威状态'  # 在条件满足时给出实际工具调用候选。

In [4]:
decisions = [(row['id'],) + decide_need_tool(row) for row in requests]  # 对六条事件运行字段驱动的判定器。
print('id | 决策 | 原因')  # 输出可审计结果表标题。
for item in decisions:  # 逐条展示决策与证据。
    print(item[0], item[1], item[2])  # 输出当前事件的控制面判断。
print('调用工具数:', sum(item[1] == '调用工具' for item in decisions), '，直接回答数:', sum(item[1] == '直接回答' for item in decisions))  # 汇总调用与 no-tool 的分布。

id | 决策 | 原因
C01 直接回答 无实时或私有状态
C02 调用工具 需要权威状态
C03 调用工具 需要权威状态
C04 调用工具 需要权威状态
C05 直接回答 无实时或私有状态
C06 拒绝并说明授权 缺少授权
调用工具数: 3 ，直接回答数: 2


In [5]:
how_to = {'id':'C07','text':'如何取消订单','fresh':False,'private':False,'write':False,'authorized':True,'slots':True}  # 构造包含高风险词但不执行动作的反例。
wrong = '调用工具' if '取消' in how_to['text'] else '直接回答'  # 应用关键词基线得到错误行为。
fixed, reason = decide_need_tool(how_to)  # 应用字段驱动判定得到修正行为。
print('失败案例:', how_to['text'], '关键词=', wrong, '，修正=', fixed, '，原因=', reason)  # 输出错误触发与修复后的解释。
print('生产差距：线上还要接入工具健康度、数据新鲜度、租户权限和策略版本，并记录 false-call 与 miss-call。')  # 说明生产决策所需的额外状态。

失败案例: 如何取消订单 关键词= 调用工具 ，修正= 直接回答 ，原因= 无实时或私有状态
生产差距：线上还要接入工具健康度、数据新鲜度、租户权限和策略版本，并记录 false-call 与 miss-call。


In [6]:
assert decide_need_tool(requests[1])[0] == '调用工具'  # 验证实时私有物流查询会进入工具路径。
assert decide_need_tool(requests[5])[0] == '拒绝并说明授权'  # 验证无授权写操作不会调用工具。
assert fixed == '直接回答'  # 验证说明性反例不会被关键词误触发。